In [ ]:
## Packages Import
%matplotlib widget
import copy
import time
import numpy             as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

In [ ]:
class VLMPanel:
    """
    Vortex Lattice Method panel with ring vortex
    """
    #--------------------#
    #   Initialization   #
    #--------------------#
    def __init__(self, p, v, w, b):
        """
        Panel is of quadrilateral shape and vertices must be ordered from root
        position on leading-edge proceeding in counterclockwise direction.
        The pannel can be a wing pannel (with an offset ring) or a wake pannel (with the ring being the pannel)
        """
        # Wing geometry
        self.span = b       # wing span [m]

        # nature of the pannel (wing or wake)
        self.nature = w        # 0 for wing, 1 for wake

        # Panel geometry
        self.pnt    = p        # panel vertices
        self.ctr    = None     # control point
        self.normal = None     # normal direction
        self.chord  = None     # chord length [m]
        self.width  = None     # pannel width [m]
        self.area   = None     # panel area   [m**2]
        
        # Vortex parameters
        self.vrt = v            # ring vortex points
        self.bound = None       # length of the bound segment in the y axis

        if self.nature == 0 :
            self._get_panel_geom()
      
    #-------------#
    #   Methods   #
    #-------------#
    def _get_panel_geom(self):
        """
        Compute panel geometric parameters such as control point position, normal
        versor, chord length, width and area, starting from panel's vertices.
        """
        p = self.pnt
        v = self.vrt
        # Compute representative panel's geometric parameters
        c_avg = 0.5 * (np.linalg.norm(p[2] - p[1]) + np.linalg.norm(p[3] - p[0]))  # average chord length
        w_avg = 0.5 * (np.linalg.norm(p[2] - p[3]) + np.linalg.norm(p[1] - p[0]))  # average width

        # Compute position of control point
        cp = (p[0] + p[1] + 3*p[3] + 3*p[2])/8    # three-quarter line

        # Compute normal versor and panel surface
        ai = np.zeros(3)
        for i, pi in enumerate(p):
            qi = p[(i + 1) % len(p)]
            ai += np.cross(pi, qi)
        av = 0.5 * ai
        a = np.linalg.norm(av)
        if a > 0.0:
            n = av / a
        else:
            #Degenerate cells with zero area
            print(f"Degenerate panel {i} with {len(p)} vertices")

        b = v[1] - v[0]

        self.bound  = b[2]
        self.ctr    = cp
        self.normal = n
        self.chord  = c_avg
        self.width  = w_avg
        self.area   = a


In [ ]:
class VLMSolver:
    """
    Vortex Lattice Method solver
    """
    #--------------------#
    #   Initialization   #
    #--------------------#
    def __init__(self, b, c, alpha, lamb, delta, sym,
                u_inf, n, m):
        # Wing geometry
        self.span     = b       # wing span                 [m]
        self.chord    = c       # chord lengths             [m]
        self.aoa      = alpha   # angle of attack           [rad]
        self.sweep    = lamb    # quarter-chord sweep angle [rad]
        self.dihedron = delta   # dihedral angle            [rad]
        self.symmetric = sym    # symmetric wing

        # Flow properties
        self.U = u_inf      # inflow velocity [m/s]

        # Discretization
        self.N = n                  # number of panels in spanwise direction
        self.M = m                  # number of panels in chordwise direction
        self.wing_panels = None     # wing panel objects
        self.wing        = None     # wing ring corner points
        self.wake_panels = None     # wake panel objects
        self.wake        = None     # wake corner points

        # Linear system
        self.A = None       # coefficient matrix
        self.b = None       # right hand side
    
    #-------------#
    #   Methods   #
    #-------------#
    def _unit_rot(self, v, theta, ax):
        """
        Elemental clockwise rotation of the reference system axes.
        
        Input:
        v     -> vector componets - shape: (N, 3)
        theta -> rotation angle - clockwise rotation: theta>0
        ax    -> rotation axis index
        """
        # Assemble rotation matrix
        rot = np.zeros((3, 3))  # rotation matrix
        # diagonal elements
        rot[ax, ax] = 1.0
        rot[ax-1, ax-1] = np.cos(theta)
        rot[ax-2, ax-2] = np.cos(theta)
        # extra-diagonal elements
        rot[ax-2, ax-1] = -np.sin(theta)
        rot[ax-1, ax-2] = np.sin(theta)

        # Apply rotation
        v_rot = np.sum(rot[None, :, :] * v[:, None, :], axis=2)
        return v_rot


    def _build_mesh(self):
        """
        Discretization of the wing by a finite number of lattices.
        """
        m = self.M
        n = self.N
        b     = self.span
        alpha = self.aoa
        lamb  = self.sweep
        delta = self.dihedron
        c     = self.chord
        # Leading-edge length of the semi-wing
        le = (0.5 * b) /  np.cos(delta)
        s_min, s_max = 0.0, le
        # Spanwise wing discretization
        s = np.linspace(s_min, s_max, (n+1))    # leading-edge coordinates
        s = np.tile(s,m+1)
        # Chordwise wing panels discretiration
        i = np.arange(m+1)[:, None]  
        j = c[None, :]                
        ch = (i/m) * j               
        ch = ch.ravel()              
        # Change panels corner points of frame
        p_wing  = np.column_stack((ch, np.zeros_like(s), s))                                          # wing frame
        p_swept = np.column_stack((p_wing[:,2]*np.tan(lamb)+p_wing[:,0],p_wing[:,1],p_wing[:,2]))     # swept frame
        p_yaw   = self._unit_rot(p_swept, -delta, 0)                                                  # yawed frame
        p_earth = self._unit_rot(p_yaw,  -alpha, 2)                                                   # earth frame
        # Ring vertices
        r_wing = copy.copy(p_wing)
        for j in range(n+1):
            for i in range(m):
                r_wing[i*(n+1)+j][0] += (r_wing[(i+1)*(n+1)+j][0]-r_wing[i*(n+1)+j][0]) * 0.25        # one quarter chord offset
            r_wing[m*(n+1)+j][0] += (r_wing[m*(n+1)+j][0]-r_wing[(m-1)*(n+1)+j][0])/3
        # Change of frame
        r_swept = np.column_stack((r_wing[:,2]*np.tan(lamb)+r_wing[:,0],r_wing[:,1],r_wing[:,2]))     # swept frame
        r_yaw   = self._unit_rot(r_swept, -delta, 0)                                                  # yawed frame
        r_earth = self._unit_rot(r_yaw,  -alpha, 2)                                                   # earth frame
        # take care of the symmetry
        if self.symmetric :
            x_sym, y_sym, z_sym = np.array([]), np.array([]), np.array([])
            for i in range(m+1):
                x_sym = np.concatenate([x_sym, p_earth[i*(n+1):(i+1)*(n+1),0][::-1],p_earth[i*(n+1)+1:(i+1)*(n+1):,0]])
                y_sym = np.concatenate([y_sym, p_earth[i*(n+1):(i+1)*(n+1),1][::-1],p_earth[i*(n+1)+1:(i+1)*(n+1):,1]])
                z_sym = np.concatenate([z_sym, -p_earth[i*(n+1):(i+1)*(n+1),2][::-1],p_earth[i*(n+1)+1:(i+1)*(n+1):,2]])
            p_earth = np.column_stack((x_sym, y_sym, z_sym))
            x_sym, y_sym, z_sym = np.array([]), np.array([]), np.array([])
            for i in range(m+1):
                x_sym = np.concatenate([x_sym, r_earth[i*(n+1):(i+1)*(n+1),0][::-1],r_earth[i*(n+1)+1:(i+1)*(n+1):,0]])
                y_sym = np.concatenate([y_sym, r_earth[i*(n+1):(i+1)*(n+1),1][::-1],r_earth[i*(n+1)+1:(i+1)*(n+1):,1]])
                z_sym = np.concatenate([z_sym, -r_earth[i*(n+1):(i+1)*(n+1),2][::-1],r_earth[i*(n+1)+1:(i+1)*(n+1):,2]])
            r_earth = np.column_stack((x_sym, y_sym, z_sym))
            self.N  = 2*n
            n       = self.N
        self.wing = r_earth
        panels = []     # list of panel objects
        for i in range(m):
            for j in range(n):
                pnt = [p_earth[i*(1+n)+j],
                    p_earth[i*(n+1)+j+1],
                    p_earth[(i+1)*(n+1)+j+1],
                    p_earth[(i+1)*(n+1)+j]]    # ordered panel vertices    
                vtx = [r_earth[i*(1+n)+j],
                    r_earth[i*(n+1)+j+1],
                    r_earth[(i+1)*(n+1)+j+1],
                    r_earth[(i+1)*(n+1)+j]]    # ordered ring vertices
                panels.append(VLMPanel(p=pnt, v=vtx, w=0, b=b))
        self.wing_panels = panels
        self.wake_panels = []
        self.wake        = []

    def _induced_velocity (self,p,c1,c2,gamma):
        """
        Return the velocity induced by the vortex segments [c1, c2], of strentgh gamma,
        at the point p
        """
        r1 = p[:, None, :] - c1[None, :, :]    # (N,M,3)
        r2 = p[:, None, :] - c2[None, :, :]    # (N,M,3)
        r0 = c2[None, :, :] - c1[None, :, :]   # (1,M,3)
        r1_norm = np.linalg.norm(r1, axis=2)   # (N,M)
        r2_norm = np.linalg.norm(r2, axis=2)
        cross = np.cross(r1, r2)                # (N,M,3)
        cross_norm2 = np.sum(cross**2, axis=2)  # (N,M)
        eps = 1e-12
        r1_norm_safe = np.where(r1_norm < eps, 1.0, r1_norm)
        r2_norm_safe = np.where(r2_norm < eps, 1.0, r2_norm)
        cross_norm2_safe = np.where(cross_norm2 < eps, 1.0, cross_norm2)
        mask = ((r1_norm > eps) &(r2_norm > eps) &(cross_norm2 > eps))  # if one point is on the segment, its induced velocity is 0
        # terme entre parenthèses
        term = np.sum(r0 * (r1 / r1_norm_safe[:, :, None] - r2 / r2_norm_safe[:, :, None]), axis=2)  # (N,M)
        coeff = gamma[None, :] / (4 * np.pi)
        v = coeff[:, :, None] * (cross / cross_norm2_safe[:, :, None]) * term[:, :, None]
        v[~mask] = 0
        return np.sum(v, axis=1)  # (N,3)
    
    def _vectorize (self, grid, gamma) :
        """
        Transform the grid corner points into two array of points that define all
        the segments, needed for the _induced_velocity function, and the circulation
        of each segment
        """
        n = self.N
        m = round(np.size(grid)/(3*(n+1)))-1
        idx = np.arange((m+1)*(n+1)).reshape(m+1, n+1)          # reshape for building the segments points
        c1_j = grid[idx[:, :-1]]                                # spanwise segments
        c2_j = grid[idx[:, 1:]]
        c1_i = grid[idx[:-1, :]]                                # chordwise segments
        c2_i = grid[idx[1:, :]]
        c1 = np.concatenate([c1_j.reshape(-1,3), c1_i.reshape(-1,3)])
        c2 = np.concatenate([c2_j.reshape(-1,3), c2_i.reshape(-1,3)])
        gamma = gamma.reshape(m, n)
        gamma_seg_j =  gamma[1:, :] - gamma[:-1, :]
        gamma_seg_j_full = np.vstack([gamma[0, :], gamma_seg_j, -gamma[-1, :]])  # shape (m+1, n)
        gamma_seg_i =  -gamma[:, 1:] + gamma[:, :-1]
        gamma_seg_i_full = np.hstack([-gamma[:, [0]], gamma_seg_i, gamma[:, [-1]]])  # shape (m, n+1)
        gamma_seg = np.concatenate([gamma_seg_j_full.ravel(), gamma_seg_i_full.ravel()])
        return c1, c2, gamma_seg

    def _build_A (self):
        """
        Construct the influence coefficients matrix A
        """
        m, n = self.M, self.N
        A = np.zeros((n*m,n*m))
        for i, ni in enumerate(self.wing_panels):
            for j, vj in enumerate(self.wing_panels):
                v = vj.vrt
                c1 = np.array([v[0], v[1], v[2], v[3]]).reshape(-1,3)
                c2 = np.array([v[1], v[2], v[3], v[0]]).reshape(-1,3)
                p  = np.array([ni.ctr]).reshape(1,3)
                v_ring =  self._induced_velocity(p, c1, c2, np.array([1,1,1,1]))[0]
                A[i][j] = np.dot(v_ring, ni.normal)
                self.A =A

    def _build_b (self, gamma_wake):
        """ 
        Construct the RHS b

        Input :
            -> gamma_wake the vortex strentgh of each wake ring
        """
        m, n  = self.M, self.N
        wake  = self.wake
        u_inf = self.U
        u_inf = np.tile(u_inf, (m*n, 1))
        normal = np.array([ni.normal for ni in self.wing_panels])
        b = np.zeros(m*n)
        if np.size(wake) == 0 :
            v = np.tile(np.array([0,0,0]), (m*n,1))
        else :
            ctrl   = np.array([ni.ctr for ni in self.wing_panels])
            c1, c2, gamma_seg = self._vectorize(wake, gamma_wake)
            v = self._induced_velocity(ctrl, c1, c2, gamma_seg)
        b = -np.sum((v + u_inf) * normal, axis=1)
        self.b = b
    
    def _lift (self, gamma):
        """
        Compute the total lift of the wing
        """
        n     = self.N
        u_inf = self.U
        L = 0
        for k, pk in enumerate(self.wing_panels):
            if k < n :          # Leading edge computation
                L += gamma[k]*pk.bound
            else :
                L += (gamma[k] - gamma[k-n])*pk.bound
        return -L*u_inf[0]
        
    def _time_sim(self, t, dt):
        """ 
        Do the time stepping simulation with the wake relaxation
        """
        # Initialization
        u_inf = self.U
        m, n  = self.M, self.N
        # we memorize the wake ring strentgh
        gamma_wake = []     
        self._build_mesh()
        n = self.N
        wing = self.wing
        self._build_A()
        A = self.A
        inv_A = np.linalg.inv(A)
        self._build_b(gamma_wake)
        b = self.b
        Gamma = inv_A @ b                  
        Te  = wing[-(n+1):]                    # get the trailing edge for the shedding
        self.wake = Te
        for s in range(round(t/dt)):
            # simulate the wing advancement
            wake = self.wake + dt*u_inf
            # shed one row of ring
            wake = np.concatenate([Te, wake])
            gamma_wake = np.concatenate([Gamma[-(n):], gamma_wake])
            self.wake = wake
            # update the right hand side and gamma copmutation
            self._build_b(gamma_wake)
            Gamma = inv_A @ self.b 
            # simulate the wake rollup
            grid      = np.concatenate([wing[:m*(n+1)], wake])
            gamma_tot = np.concatenate([Gamma, gamma_wake])
            c1, c2, gamma_v = self._vectorize(grid, gamma_tot)
            self.wake = wake + np.concatenate([np.zeros_like(Te), dt*self._induced_velocity(wake[n+1:], c1, c2, gamma_v)])
        # build wake_panels, usefull for plotting
        panels = []
        for i in range(round(len(self.wake)/(n+1))-1):
            for j in range(n):
                pnt = [wake[i*(1+n)+j],
                    wake[i*(n+1)+j+1],
                    wake[(i+1)*(n+1)+j+1],
                    wake[(i+1)*(n+1)+j]]    # ordered panel vertices  
                panels.append(VLMPanel(p=pnt, v=pnt, w=1, b=b))
        self.wake_panels = panels

In [ ]:
## Utilities

def chord_fn(n, c0, b, delta, shape="elliptical"):
    """
    Return chord length distribution along spanwise direction according to the
    desired planform shape.

    Input:
        n     -> number of spanwise pannels
        c0    -> root chord length
        b     -> wing span
        delta -> dihedral angle
        shape -> wing planform
    """
    match shape:
        case "rectangular":
            c = c0 * np.ones(n)
        case "elliptical":
            le = (0.5 * b) /  np.cos(delta)
            s_min, s_max = 0.0, le
            s = np.linspace(s_min, s_max, n)
            c = c0 * np.sqrt(1 - (s/b)**2)
    return c




In [ ]:
## Parameters Definition

# Wing geometry
B = 1         # wing span                 [m]
AR = 4        # aspect ration             [-]
ALPHA  = 8   # angle of attack            [deg] - positive definite for counterclockwise rotations about z-axis
LAMBDA = 25  # quarter-chord sweep angle [deg] - positive definite for counterclockwise rotations about x-axis  Y-AXIS
DELTA  = 10  # dihedral angle            [deg] - positive definite for rotations oriented towards positive y-axis  X-AXIS, négatif pour dyhedre classique 

SYM = True   # symmetric wing configuration

# Flow properties
U = 1.0     # inflow velocity [m/s]

# Wing discretization
N = 10       # number of panels in spanwise direction
M = 5       # number of panels in chordwise direction

# Time simulation parameters
T  = 5   # length of the simulation in s
DT = 0.2       # time step lentgh 

# Untapered wing (constant chord length distribution)
c0 = B / AR     # chord length [m]



In [ ]:
## Time stepping algorithm 
vlm = VLMSolver(b=B,
                c=chord_fn((N+1), c0, B, DELTA),
                alpha=np.deg2rad(ALPHA),
                lamb=np.deg2rad(LAMBDA),
                delta=np.deg2rad(DELTA),
                sym=SYM,
                u_inf=np.array([U, 0.0, 0.0]),
                n=N,m=M)
#vlm._build_mesh()
vlm._time_sim(t=T, dt=DT)

# plot
fig_3d = plt.figure(figsize=(14, 8),constrained_layout=True)
ax4 = fig_3d.add_subplot(111, projection='3d')
for i, panel in enumerate(vlm.wing_panels):
        pnt    = copy.copy(panel.pnt)
        vrt    = copy.copy(panel.vrt)
        # Close the polygon shape
        pnt.append(pnt[0])
        pnt_plt = np.array(pnt)
        vrt.append(vrt[0])
        vrt_plt = np.array(vrt)

        # Plot lattice
        if i==0 :               # In order to have just one legend
                # 3D plot
                ax4.plot(pnt_plt[:, 0], pnt_plt[:, 1], pnt_plt[:, 2], color='black', label='Wing panels')
                ax4.plot(vrt_plt[:, 0], vrt_plt[:, 1], vrt_plt[:, 2], color='red', lw=0.8, label='Vortex rings')
        else :
                # 3D plot
                ax4.plot(pnt_plt[:, 0], pnt_plt[:, 1], pnt_plt[:, 2], color='black')
                ax4.plot(vrt_plt[:, 0], vrt_plt[:, 1], vrt_plt[:, 2], color='red', lw=0.8)
for i, panel in enumerate(vlm.wake_panels):
        pnt    = copy.copy(panel.pnt)
        # Close the polygon shape
        pnt.append(pnt[0])
        pnt_plt = np.array(pnt)
        # Plot lattice
        if i==0 :               # In order to have just one legend
                # 3D plot
                ax4.plot(pnt_plt[:, 0], pnt_plt[:, 1], pnt_plt[:, 2], color='blue', lw=0.8, label='Wake panels')
        else :
                # 3D plot
                ax4.plot(pnt_plt[:, 0], pnt_plt[:, 1], pnt_plt[:, 2], color='blue', lw=0.8)      
ax4.view_init(elev=15, azim=-140)
x_lim = ax4.get_xlim3d()
y_lim = ax4.get_ylim3d()
z_lim = ax4.get_zlim3d()
ax4.set_xlim(x_lim[0],x_lim[1])
ax4.set_ylim(y_lim[1],y_lim[0]) 
ax4.set_zlim(z_lim[1],z_lim[0])
ax4.set_box_aspect([-x_lim[0]+x_lim[1],-y_lim[0]+y_lim[1],-z_lim[0]+z_lim[1]])
ax4.set_xlabel("x", fontstyle='italic')
ax4.set_ylabel("y", fontstyle='italic')
ax4.set_zlabel("z", fontstyle='italic')
ax4.yaxis.set_major_locator(MultipleLocator(0.1))
ax4.set_title('3D view')
ax4.legend()